**Cell 1**

# NB16 — Llama-2-7B DPO experts on four preference datasets, seed 137

**Purpose.** Is the near-identity, floor-collapsing `R` a property of HelpSteer2, or does it
appear on every standard preference dataset? NB16 trains 14 DPO experts with the NB14 recipe
on four datasets and computes one relationship matrix `R` per dataset (4 matrices).

| dataset | Hugging Face | experts | pair source |
|---|---|---|---|
| `hh_rlhf` | Anthropic/hh-rlhf | helpful, harmless, humor | human chosen/rejected (helpful-base, harmless-base); humor: helpful-base pairs ordered by the humor classifier |
| `pku_saferlhf` | PKU-Alignment/PKU-SafeRLHF | better, safer | the same response pairs, oriented by `better_response_id` resp. `safer_response_id` |
| `helpsteer1` | nvidia/HelpSteer | helpfulness, correctness, coherence, complexity, verbosity | NB12 builder: within-prompt, higher rating chosen, ties discarded |
| `ultrafeedback` | openbmb/UltraFeedback | helpfulness, honesty, instruction_following, truthfulness | within-instruction over the 4 completions, higher aspect rating chosen, ties and N/A discarded |

**Humor.** HH-RLHF has no humor labels. As in the Rewarded-Soups / RiC assistant task, humor is
defined by the classifier `mohameddhiab/humor-no-humor`: both responses of each helpful-base pair
get P(HUMOR), pairs whose scores differ by at least `HUMOR_MARGIN` are kept, the funnier one is
chosen. The classifier only orders pairs; it is never used during training. Caveat for the
thesis: evaluating humor with the same classifier later would be circular.

**Pair selection (NB12 rule).** Exactly one pair per prompt. Within a prompt, and when a larger
pool is cut to `N` prompts, the ranking is: largest target gap, then higher chosen rating, then
smallest gap on the other attributes, then a seed-137 hash (`select_pairs` of
`scripts/0_dpo_expert_one_pair.py`, unchanged). Gap = rating difference (HelpSteer1,
UltraFeedback) or P(HUMOR) difference (humor); PKU-SafeRLHF and HH helpful/harmless have binary
labels, so only the seeded hash decides. HelpSteer1 uses the NB12 code and dataset revision, so
its five experts see exactly the NB12 pairs: a direct TinyLlama (NB12) vs Llama-2-7B comparison.

**Number of pairs `N`.** Equal for all experts of a dataset, so that norms within one `R` are
comparable (the argument of Section 5.2), and as in NB12 set by the binding objective: by default
`N_d` = the smallest number of *eligible prompts* (prompts with at least one non-tied pair) in
dataset d. HelpSteer1 gives 5,085 (complexity), as in NB12. Cell 15 prints all counts, `N_d` and
the expected training time before anything is trained. `N_OVERRIDE` in Cell 9 sets `N` by hand,
`MAX_PAIRS_CAP` caps all datasets, `SAME_N_ALL_DATASETS = True` uses one common `N`. Training
longer than `MAX_HOURS_WITHOUT_CONFIRM` only starts with `CONFIRM_LONG_RUN = True`.

**Recipe (identical to NB14).** Raw `meta-llama/Llama-2-7b-hf`, prompt
`\n\nHuman: {prompt} \n\nAssistant:` (HH-RLHF multi-turn: original context, then
` \n\nAssistant:`), slow tokenizer, LoRA r=8, alpha=16, dropout 0.05 on all seven projections,
DPO beta 0.1, sigmoid loss, 1 epoch, lr 5e-4 cosine with 3 % warmup, effective batch 8 (1 × 8),
max length 512 / prompt 256, bf16, fixed final checkpoint, seed 137, pair seed 137.
Script: `scripts/0_dpo_expert_multidata.py` (new, additive; NB14's script is imported unchanged).

**Expected outcome from the theory.** For m = 2 (PKU-SafeRLHF) the floor of `R_cos` always
collapses, because `R_cos⁻¹·1 = 1/(1+c)·1 > 0`; only `R_gram` with very unequal norms could open.
For m = 3, 4, 5 the floor opens only if one expert is redundant (closer to the others than they
are to each other).

**Requirements.** A100 GPU, Hugging Face token with Llama-2 access as Colab secret `HF_TOKEN`.
If PKU-SafeRLHF asks for access, accept its terms on the dataset page with the same account.

**Runtime (estimate).** About 0.6 s per pair on an A100 (NB14: 2,690 pairs in 0.46 h), i.e.
about 0.9 h per expert at `N = 5,085`; Cell 15 prints the total for the chosen `N`, plus about
20 min for data loading and humor scoring. Run all is supported and resumable: after a
disconnect, run all again. Finished experts are verified and skipped, an interrupted one resumes
from its last checkpoint on Drive.

**This run.** UltraFeedback is left out for now (`DATASET_ORDER` in Cell 9; its 4 experts would add
about 37 h). `N` is fixed to the counts of the pair plan: HelpSteer1 5,085, HH-RLHF 15,903 (humor
margin 0.5), PKU-SafeRLHF 38,636; 10 experts, about 26 GPU hours. The cheapest dataset trains
first, and each dataset's `R` is computed and copied to Drive as soon as its last expert is done,
so the matrices arrive one by one (HelpSteer1 after about 4.5 h, HH-RLHF after about 13 h).

**Outputs.** The adapters on Drive; per dataset `R_cos`, `R_gram`, norms, floor LP, verdict;
training diagnostics; all in `results/nb16_.../geometry`.


**Cell 2**

Mount Drive, then clone or update the repository.


In [ ]:
# Cell 3
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import os, shutil, sys
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis
for required in ("scripts/0_dpo_expert_multidata.py", "scripts/0_dpo_expert_llama2.py", "scripts/0_dpo_expert_one_pair.py",
                 "scripts/0_dpo_expert.py", "scripts/assistant_adapter_geometry.py"):
    assert os.path.isfile(required), f"Push {required} to the repository first."


**Cell 4**

GPU check. bf16 is required; the batch shape is chosen for 40 GB.


In [ ]:
# Cell 5
!nvidia-smi
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select an A100 runtime."
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)}, {VRAM_GIB:.1f} GiB")
assert VRAM_GIB >= 38, "Llama-2-7B DPO needs an A100 (40 or 80 GB)."


**Cell 6**

Pinned NB11/NB14 DPO stack. `USE_TF=0` avoids the Colab TensorFlow/protobuf conflict.


In [ ]:
# Cell 7
import importlib.metadata, subprocess
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
print(f"Python {sys.version.split()[0]} ({sys.executable})")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "protobuf==5.29.5", "transformers==4.45.2",
    "tokenizers==0.20.3", "peft==0.13.2", "accelerate==1.1.1", "trl==0.11.4",
    "datasets==3.1.0", "huggingface_hub==0.36.0", "bitsandbytes", "pyyaml", "safetensors",
    "sentencepiece", "scipy==1.14.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages], check=True)
expected = {"transformers": "4.45.2", "trl": "0.11.4"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, f"Incompatible DPO stack: {installed}. Restart the runtime and run all again."
print({n: importlib.metadata.version(n) for n in ("transformers", "tokenizers", "peft", "accelerate", "trl", "datasets")})


**Cell 8**

Frozen settings. Everything except the datasets is the NB14 recipe.

* `DATASET_ORDER`: datasets of this run, cheapest first; add `"ultrafeedback"` to train it later.
* `N_OVERRIDE`: fixed `N` per dataset; a dataset missing here gets the automatic value
  (smallest number of eligible prompts in the dataset, as in NB12).
* `MAX_PAIRS_CAP`: optional upper bound for every dataset (`None` = no cap).
* `SAME_N_ALL_DATASETS = True`: one common `N` (the smallest `N_d`) for all 14 experts.
* `HUMOR_MARGIN`: minimum P(HUMOR) difference for a humor pair (0.5 for now).
* `CONFIRM_LONG_RUN`: training whose estimate exceeds `MAX_HOURS_WITHOUT_CONFIRM` starts only if True.


In [ ]:
# Cell 9
from pathlib import Path
import json
PROJECT_ROOT = Path("/content/master-thesis").resolve()
BASE_MODEL = "meta-llama/Llama-2-7b-hf"
RUN_TAG = "nb16_llama2_7b_multidata_dpo_seed137"
DRIVE_ROOT = Path("/content/drive/MyDrive/master-thesis-nb16") / RUN_TAG
TRAINED_ROOT = DRIVE_ROOT / "trained" / "seed_137"              # training writes here directly
RESULTS_DIR = PROJECT_ROOT / "results" / RUN_TAG / "geometry"
for directory in (DRIVE_ROOT, TRAINED_ROOT, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATASET_ORDER = ["helpsteer1", "hh_rlhf", "pku_saferlhf"]    # cheapest first; "ultrafeedback" left out for now
N_OVERRIDE = {"helpsteer1": 5085, "hh_rlhf": 15903, "pku_saferlhf": 38636}   # pair plan of 12 Oct 2026
MAX_PAIRS_CAP = None
SAME_N_ALL_DATASETS = False
HUMOR_MARGIN = 0.5
SECONDS_PER_PAIR = 0.62                # NB14: 2,690 pairs in 0.46 h on an A100
MAX_HOURS_WITHOUT_CONFIRM = 12
CONFIRM_LONG_RUN = True                # about 26 h, confirmed

SEED = 137
PAIR_SEED = 137
DPO_BETA = 0.1
EPOCHS = 1.0
LEARNING_RATE = 5e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
SAVE_STEPS = 20
RESPONSE_SPLIT = "\n\nAssistant:"      # Instructions.response_split of the supervisor's utils.py
RUN_SMOKE_TEST = True
DOWNLOAD_ADAPTER_BUNDLE = False        # about 1 GB; the bundle is always written to Drive

import importlib.util
def load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / "scripts" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

multi = load_script("nb16_dpo_script", "0_dpo_expert_multidata.py")
nb11 = multi.nb11
OBJECTIVES = {key: list(multi.objectives(key)) for key in DATASET_ORDER}
EXPERTS = [(key, objective) for key in DATASET_ORDER for objective in OBJECTIVES[key]]
print(f"{len(EXPERTS)} experts:", ", ".join(f"{k}/{o}" for k, o in EXPERTS))
print(f"training root (Drive) = {TRAINED_ROOT}\nresults (repo)        = {RESULTS_DIR}")


**Cell 10**

Hugging Face access and frozen revisions. Base model, humor classifier and all four datasets are resolved once and stored on Drive, so every expert and every rerun uses the same data and weights.


In [ ]:
# Cell 11
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add your Hugging Face token as Colab secret HF_TOKEN."
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

revisions_path = DRIVE_ROOT / "revisions.json"
if revisions_path.is_file():
    REV = json.loads(revisions_path.read_text())
else:
    api = HfApi()
    REV = {"base_model": api.model_info(BASE_MODEL).sha, "humor_model": api.model_info(multi.HUMOR_MODEL).sha,
           "datasets": {}}
    for key in DATASET_ORDER:
        name = multi.DATASETS[key]["hf_name"]
        try:
            REV["datasets"][key] = multi.PINNED_REVISIONS.get(key) or api.dataset_info(name).sha
        except Exception as error:
            raise RuntimeError(f"Cannot access {name}. If it is gated, accept its terms on "
                               f"https://huggingface.co/datasets/{name} with the HF_TOKEN account.") from error
    revisions_path.write_text(json.dumps(REV, indent=2))
BASE_REVISION = REV["base_model"]
assert BASE_REVISION and len(BASE_REVISION) == 40
print(json.dumps(REV, indent=2))


**Cell 12**

Humor scores. Every unique final response of the HH-RLHF helpful-base pairs gets P(HUMOR) from
`mohameddhiab/humor-no-humor` (DistilBERT, a few minutes on the GPU). The scores are stored on
Drive and bound to the humor expert by their SHA-256. The table shows how many pairs survive at
several margins; with very few pairs, lower `HUMOR_MARGIN` in Cell 9 before training starts.


In [ ]:
# Cell 13
import numpy as np, pandas as pd
HUMOR_PATH = DRIVE_ROOT / "humor_scores_hh_helpful_base.json"
hh_helpful_rows = multi.load_objective_rows("hh_rlhf", "humor", REV["datasets"]["hh_rlhf"])
if HUMOR_PATH.is_file():
    humor_payload = json.loads(HUMOR_PATH.read_text())
    assert humor_payload["humor_model_revision"] == REV["humor_model"], "humor model revision changed"
    assert humor_payload["dataset_revision"] == REV["datasets"]["hh_rlhf"], "HH-RLHF revision changed"
    print(f"[reuse] {HUMOR_PATH}")
else:
    texts = multi.humor_texts(hh_helpful_rows)
    print(f"scoring {len(texts):,} unique responses ...")
    scores = multi.score_humor(texts, revision=REV["humor_model"], batch_size=256)
    humor_payload = {"humor_model": multi.HUMOR_MODEL, "humor_model_revision": REV["humor_model"],
                     "dataset": "Anthropic/hh-rlhf", "subset": "helpful-base",
                     "dataset_revision": REV["datasets"]["hh_rlhf"], "n_texts": len(texts), "scores": scores}
    HUMOR_PATH.write_text(json.dumps(humor_payload))
HUMOR_SCORES = humor_payload["scores"]

triples, dropped = multi.parse_hh_rows(hh_helpful_rows)
diff = np.array([abs(HUMOR_SCORES[multi.text_sha256(a)] - HUMOR_SCORES[multi.text_sha256(b)]) for _, a, b in triples])
p_humor = np.array(list(HUMOR_SCORES.values()))
print(f"helpful-base pairs: {len(triples):,} (dropped {dropped}); responses with P(HUMOR) > 0.5: {(p_humor > 0.5).mean():.1%}")
display(pd.DataFrame({"margin": [0.05, 0.1, 0.2, 0.3, 0.5],
                      "humor pairs": [int((diff >= m).sum()) for m in (0.05, 0.1, 0.2, 0.3, 0.5)]}).set_index("margin").T)
print(f"current HUMOR_MARGIN = {HUMOR_MARGIN}")
del triples, diff


**Cell 14**

Pair plan, counted as in NB12: candidate pairs and eligible prompts (prompts with at least one
non-tied pair = largest possible `N`) for every expert, then `N` per dataset, the selected pair
subsets (hashes), the expected training time and one example prompt per dataset. HelpSteer1 is
checked against the NB12 counts. **Check this table before the long run.**


In [ ]:
# Cell 15
_rows_cache = {}
def rows_for(key, objective):
    cache_key = (key, multi.HH_SUBSET[objective] if key == "hh_rlhf" else None)
    if cache_key not in _rows_cache:
        _rows_cache[cache_key] = multi.load_objective_rows(key, objective, REV["datasets"][key])
    return _rows_cache[cache_key]

RANK_FIELDS = ("pair_id", "raw_prompt", "target_rating_gap", "chosen_target_rating", "other_axes_abs_gap_sum")
CANDIDATE_COUNTS, ELIGIBLE_PROMPTS, EXAMPLES, _ranked = {}, {}, {}, {}
for key in DATASET_ORDER:
    for objective in OBJECTIVES[key]:
        candidates = multi.build_pairs(key, objective, rows_for(key, objective),
                                       humor_scores=HUMOR_SCORES, humor_margin=HUMOR_MARGIN)
        CANDIDATE_COUNTS[(key, objective)] = len(candidates)
        ELIGIBLE_PROMPTS[(key, objective)] = multi.eligible_prompts(candidates)
        EXAMPLES.setdefault(key, candidates[0] if candidates else None)
        # Only the ranking fields are kept; select_pairs on them gives the same subset as in the script.
        _ranked[(key, objective)] = [{f: p[f] for f in RANK_FIELDS} for p in candidates]
        del candidates
    print(f"[pairs] {key}: " + ", ".join(f"{o}={CANDIDATE_COUNTS[(key, o)]:,} pairs / {ELIGIBLE_PROMPTS[(key, o)]:,} prompts"
                                        for o in OBJECTIVES[key]), flush=True)
_rows_cache.clear()
if "helpsteer1" in DATASET_ORDER and REV["datasets"]["helpsteer1"] == multi.PINNED_REVISIONS["helpsteer1"]:
    for objective in OBJECTIVES["helpsteer1"]:
        observed = (CANDIDATE_COUNTS[("helpsteer1", objective)], ELIGIBLE_PROMPTS[("helpsteer1", objective)])
        expected = (multi.NB12_CANDIDATES[objective], multi.NB12_ELIGIBLE_PROMPTS[objective])
        assert observed == expected, f"HelpSteer1 {objective}: {observed} differs from NB12 {expected}"
    print("[OK] HelpSteer1 counts equal NB12")

def max_common_n(key):
    return min(ELIGIBLE_PROMPTS[(key, o)] for o in OBJECTIVES[key])

N_PAIRS = {}
for key in DATASET_ORDER:
    n = int(N_OVERRIDE.get(key) or max_common_n(key))
    N_PAIRS[key] = min(n, int(MAX_PAIRS_CAP)) if MAX_PAIRS_CAP else n
if SAME_N_ALL_DATASETS:
    common = min(N_PAIRS.values())
    N_PAIRS = {key: common for key in DATASET_ORDER}
for key, n in N_PAIRS.items():
    assert n <= max_common_n(key), f"N for {key} = {n} exceeds its eligible prompts {max_common_n(key)}"
    if n < 1000:
        print(f"[warning] {key}: only N = {n} pairs per expert; specialisation will be weak.")

PAIR_ID_HASHES = {}
for key, objective in EXPERTS:
    selected = multi.select_pairs(_ranked[(key, objective)], seed=PAIR_SEED, max_pairs=N_PAIRS[key])
    PAIR_ID_HASHES[(key, objective)] = nb11.canonical_hash({"pair_ids": [p["pair_id"] for p in selected]})
del _ranked

PLAN_DF = pd.DataFrame([{"dataset": k, "objective": o, "candidate_pairs": CANDIDATE_COUNTS[(k, o)],
                         "eligible_prompts": ELIGIBLE_PROMPTS[(k, o)], "N": N_PAIRS[k],
                         "hours (est.)": N_PAIRS[k] * SECONDS_PER_PAIR / 3600,
                         "pairs_sha256": PAIR_ID_HASHES[(k, o)][:16] + "..."} for k, o in EXPERTS])
display(PLAN_DF.round(2))
ESTIMATED_HOURS = float(PLAN_DF["hours (est.)"].sum())
(DRIVE_ROOT / "pair_plan.json").write_text(json.dumps(
    {"n_pairs": N_PAIRS, "humor_margin": HUMOR_MARGIN, "cap": MAX_PAIRS_CAP, "same_n_all": SAME_N_ALL_DATASETS,
     "selection": "one pair per prompt (NB12 select_pairs)", "estimated_hours": ESTIMATED_HOURS,
     "rows": [{"dataset": k, "objective": o, "candidate_pairs": CANDIDATE_COUNTS[(k, o)],
               "eligible_prompts": ELIGIBLE_PROMPTS[(k, o)], "pair_ids_sha256": PAIR_ID_HASHES[(k, o)]}
              for k, o in EXPERTS]}, indent=2))
PLAN_DF.to_csv(RESULTS_DIR / "pair_plan.csv", index=False)
print(f"N per dataset: {N_PAIRS}")
print(f"total training pairs: {sum(N_PAIRS[k] for k, _ in EXPERTS):,}; estimated training time: {ESTIMATED_HOURS:.1f} h")
if ESTIMATED_HOURS > MAX_HOURS_WITHOUT_CONFIRM and not CONFIRM_LONG_RUN:
    print(f"[STOP before training] {ESTIMATED_HOURS:.1f} h > {MAX_HOURS_WITHOUT_CONFIRM} h. Lower N (N_OVERRIDE, "
          "MAX_PAIRS_CAP) or set CONFIRM_LONG_RUN = True in Cell 9.")
for key, pair in EXAMPLES.items():
    if pair:
        print(f"\n--- {key} example prompt ---\n{multi.render_prompt(pair, RESPONSE_SPLIT)[-300:]!r}")


**Cell 16**

Verification of a finished expert against the frozen settings.


In [ ]:
# Cell 17
import hashlib
EXPECTED_LORA = {"r": 8, "alpha": 16, "dropout": 0.05, "bias": "none",
                 "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def expert_dir(key, objective):
    return TRAINED_ROOT / key / f"dpo_{objective}"

def verify_expert(key, objective):
    directory = expert_dir(key, objective)
    manifest_path = directory / "training_manifest.json"
    weights_path = directory / "adapter/adapter_model.safetensors"
    if not manifest_path.is_file() or not weights_path.is_file():
        raise FileNotFoundError(f"Incomplete expert: {directory}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {
        "completed": True, "training_method": "DPO", "dataset_key": key, "objective": objective,
        "base_model_name": BASE_MODEL, "base_revision": BASE_REVISION,
        "dataset_name": multi.DATASETS[key]["hf_name"], "dataset_revision": REV["datasets"][key],
        "dataset_split": "train", "pair_seed": PAIR_SEED, "selected_pair_count": N_PAIRS[key],
        "one_pair_per_prompt": True, "selected_unique_prompt_count": N_PAIRS[key],
        "selected_pair_ids_sha256": PAIR_ID_HASHES[(key, objective)], "response_split": RESPONSE_SPLIT,
        "beta": DPO_BETA, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM, "max_length": MAX_LENGTH,
        "max_prompt_length": MAX_PROMPT_LENGTH, "precision": "bf16", "seed": SEED,
        "reward_model_used_during_training": False,
        "checkpoint_selection": "fixed final epoch; no reward-model selection",
    }
    if objective == "humor":
        expected.update({"humor_margin": HUMOR_MARGIN, "humor_model_revision": REV["humor_model"],
                         "humor_scores_sha256": sha256_file(HUMOR_PATH)})
    for field, value in expected.items():
        if manifest.get(field) != value:
            raise RuntimeError(f"{directory} differs on {field}: {manifest.get(field)!r} != {value!r}")
    if manifest.get("lora") != EXPECTED_LORA:
        raise RuntimeError(f"{directory} has a different LoRA configuration")
    if sha256_file(weights_path) != manifest.get("adapter_model_sha256"):
        raise RuntimeError(f"Adapter hash mismatch: {directory}")
    return manifest

def is_complete(key, objective):
    try:
        verify_expert(key, objective)
        return True
    except FileNotFoundError:
        return False

STATUS = {f"{k}/{o}": is_complete(k, o) for k, o in EXPERTS}
print(f"complete: {sum(STATUS.values())}/{len(STATUS)}", STATUS)


**Cell 18**

Smoke test with 16 HH-RLHF helpful pairs (two optimizer steps, multi-turn prompt path). It runs only before the first expert is trained.


In [ ]:
# Cell 19
from collections import deque
import time

def run_logged_subprocess(command, label):
    print(f"[run] {label}", flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(command, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
    if process.wait():
        raise RuntimeError(f"{label} failed.\nLast output:\n" + "\n".join(tail))

def script_command(key, objective, output_root, max_pairs, resume=True, overwrite=False):
    command = [sys.executable, "-u", "scripts/0_dpo_expert_multidata.py",
               "--dataset_key", key, "--objective", objective, "--dataset_revision", REV["datasets"][key],
               "--base_model_name", BASE_MODEL, "--base_revision", BASE_REVISION, "--split", "train",
               "--output_root", str(output_root), "--max_pairs", str(max_pairs),
               "--response_split", RESPONSE_SPLIT, "--beta", str(DPO_BETA), "--epochs", str(EPOCHS),
               "--lr", str(LEARNING_RATE), "--batch_size", str(BATCH_SIZE), "--grad_accum", str(GRAD_ACCUM),
               "--max_length", str(MAX_LENGTH), "--max_prompt_length", str(MAX_PROMPT_LENGTH),
               "--pair_seed", str(PAIR_SEED), "--seed", str(SEED), "--save_steps", str(SAVE_STEPS)]
    if objective == "humor":
        command += ["--humor_scores", str(HUMOR_PATH), "--humor_margin", str(HUMOR_MARGIN),
                    "--humor_model_revision", REV["humor_model"]]
    if resume:
        command.append("--resume")
    if overwrite:
        command.append("--overwrite")
    return command

if RUN_SMOKE_TEST and not any(is_complete(k, o) for k, o in EXPERTS):
    smoke_root = Path("/tmp/nb16_dpo_smoke")
    shutil.rmtree(smoke_root, ignore_errors=True)
    t0 = time.perf_counter()
    run_logged_subprocess(script_command("hh_rlhf", "helpful", smoke_root, 16, resume=False, overwrite=True), "DPO smoke test")
    print(f"[OK] smoke test passed in {time.perf_counter() - t0:.0f} s (includes model download on first run)")
    shutil.rmtree(smoke_root, ignore_errors=True)
else:
    print("[skip] smoke test")


**Cell 20**

Geometry helpers, one relationship matrix per dataset: `R_cos` (primary) and `R_gram`, norms,
eigenvalues, `R⁻¹·1`, global floor LP, and the p-aware floor LP for named preferences plus 64
Dirichlet draws (seed 137). Same functions as NB15 (`scripts/assistant_adapter_geometry.py`),
which use the NB13/NB14 geometry code. Results go to the repository folder and to Drive
(`geometry/` next to the adapters), so they survive a disconnect.


In [ ]:
# Cell 21
ag = load_script("assistant_geometry", "assistant_adapter_geometry.py")
GEOMETRY_DRIVE = DRIVE_ROOT / "geometry"
GEOMETRY = {}

def verdict(summary, m, label):
    if m == 2 and label == "R_cos":
        return "m = 2: the R_cos floor always collapses (R_cos^-1 1 = 1/(1+c) 1 > 0)."
    if summary["near_singular"]:
        return "nearly singular: one expert is almost a combination of the others; interpret R^-1 1 with care."
    if not summary["inverse_R_ones_strictly_positive"]:
        return f"FLOOR OPENS; redundant axis {summary['redundant_axes_inverse_R_ones_nonpositive']}."
    if summary["n_negative_offdiag"] > 0:
        return "floor collapses although some experts are negatively aligned (conflict alone does not open it)."
    return "floor collapses with non-negative alignment (as on HelpSteer2)."

def dataset_complete(key):
    return all(is_complete(key, o) for o in OBJECTIVES[key])

def dataset_geometry(key, show=True):
    goals = OBJECTIVES[key]
    out = RESULTS_DIR / key
    out.mkdir(parents=True, exist_ok=True)
    geometry = ag.relationship_matrices({o: expert_dir(key, o) / "adapter" for o in goals})
    prefs = ag.preference_set(goals, n_dirichlet=64, seed=137)
    report = {"dataset": key, "objectives": goals, "N": N_PAIRS[key],
              "effective_update_numel": geometry["effective_update_numel"],
              "update_norms": dict(zip(goals, geometry["norms"].tolist())), "n_preferences": len(prefs)}
    for label in ("R_cos", "R_gram"):
        R = geometry[label]
        pd.DataFrame(R, index=goals, columns=goals).to_csv(out / f"{label}.csv")
        floor = ag.floor_report(R, prefs)
        floor.to_csv(out / f"floor_lp_{label}.csv", index=False)
        summary = ag.matrix_summary(R, goals)
        report[label] = {**summary, "floor_lp": {"n_preferences": len(floor), "n_collapsed": int(floor["collapsed"].sum())},
                         "verdict": verdict(summary, len(goals), label)}
    pd.DataFrame({"objective": goals, "effective_update_norm": geometry["norms"]}).to_csv(out / "update_norms.csv", index=False)
    (out / "geometry_report.json").write_text(json.dumps(report, indent=2, default=str) + "\n")
    shutil.copytree(out, GEOMETRY_DRIVE / key, dirs_exist_ok=True)
    GEOMETRY[key] = report
    if show:
        print(f"\n=== {key} (m = {len(goals)}, N = {N_PAIRS[key]}) ===")
        display(pd.DataFrame(geometry["R_cos"], index=goals, columns=goals).round(4))
        print("R_cos :", report["R_cos"]["verdict"])
        print("R_gram:", report["R_gram"]["verdict"])
    s = report["R_cos"]
    return {"dataset": key, "m": len(goals), "N": N_PAIRS[key],
            "cos offdiag min": s["offdiag_min"], "cos offdiag mean": s["offdiag_mean"],
            "cos offdiag max": s["offdiag_max"], "negative pairs": s["n_negative_offdiag"],
            "condition R_cos": s["condition_number"],
            "R_cos floor collapsed": f'{s["floor_lp"]["n_collapsed"]}/{s["floor_lp"]["n_preferences"]}',
            "R_gram floor collapsed": f'{report["R_gram"]["floor_lp"]["n_collapsed"]}/{report["R_gram"]["floor_lp"]["n_preferences"]}',
            "norm min/max": f'{geometry["norms"].min():.2f} / {geometry["norms"].max():.2f}',
            "verdict (R_cos)": s["verdict"]}

print("complete datasets:", [k for k in DATASET_ORDER if dataset_complete(k)])


**Cell 22**

Training of all missing experts, dataset by dataset. As soon as the last expert of a dataset is done, its `R` is computed, shown and copied to Drive. This is the long-running cell. Re-running it is safe: finished experts are verified and skipped, an interrupted expert resumes from its last checkpoint on Drive.


In [ ]:
# Cell 23
import gc
if ESTIMATED_HOURS > MAX_HOURS_WITHOUT_CONFIRM and not CONFIRM_LONG_RUN and not all(is_complete(k, o) for k, o in EXPERTS):
    raise RuntimeError(f"Estimated training time {ESTIMATED_HOURS:.1f} h exceeds {MAX_HOURS_WITHOUT_CONFIRM} h. "
                       "Lower N or set CONFIRM_LONG_RUN = True in Cell 9, then run all again.")
TIMES_PATH = DRIVE_ROOT / "training_wall_clock.json"
times = json.loads(TIMES_PATH.read_text()) if TIMES_PATH.is_file() else {}

number = 0
for key in DATASET_ORDER:
    for objective in OBJECTIVES[key]:
        number += 1
        label = f"{key}/{objective}"
        print(f"\n##### {number}/{len(EXPERTS)}: {label} (N = {N_PAIRS[key]}) #####")
        if is_complete(key, objective):
            print(f"[skip] verified {label}")
            continue
        t0 = time.perf_counter()
        run_logged_subprocess(script_command(key, objective, TRAINED_ROOT, N_PAIRS[key]), f"DPO {label}")
        verify_expert(key, objective)
        times[label] = time.perf_counter() - t0   # last completed segment only
        TIMES_PATH.write_text(json.dumps(times, indent=2))
        trainer_dir = expert_dir(key, objective) / "trainer"
        if trainer_dir.exists():
            shutil.rmtree(trainer_dir)          # final adapter is verified on Drive; checkpoints no longer needed
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[OK] {label}: {times[label] / 3600:.2f} h")
    if not (GEOMETRY_DRIVE / key / "geometry_report.json").is_file():
        print(f"\n[dataset complete] {key}: computing R")
        dataset_geometry(key)
print(f"[OK] all {len(EXPERTS)} experts of {DATASET_ORDER} are complete.")


**Cell 24**

Inventory and training diagnostics. `train_loss` below ln 2 = 0.693 means the expert separates
chosen from rejected better than chance; the last logged reward accuracy and margin come from
the trainer log. An expert that stays near ln 2 has learned little, which matters when its row
of `R` is interpreted (NB14 had this on the quality axes).


In [ ]:
# Cell 25
inventory = []
for key, objective in EXPERTS:
    if not is_complete(key, objective):
        continue
    manifest = verify_expert(key, objective)
    metrics = manifest.get("train_metrics", {})
    log_path = expert_dir(key, objective) / "log_history.json"
    logged = [e for e in json.loads(log_path.read_text())] if log_path.is_file() else []
    last = next((e for e in reversed(logged) if "rewards/accuracies" in e), {})
    inventory.append({
        "dataset": key, "objective": objective, "N": manifest["selected_pair_count"],
        "train_loss": metrics.get("train_loss"),
        "below_ln2": (metrics.get("train_loss") or 1) < np.log(2),
        "last_reward_accuracy": last.get("rewards/accuracies"), "last_reward_margin": last.get("rewards/margins"),
        "train_runtime_h": (metrics.get("train_runtime") or float("nan")) / 3600,
        "adapter_model_sha256": manifest["adapter_model_sha256"],
    })
INVENTORY_DF = pd.DataFrame(inventory)
INVENTORY_DF.to_csv(RESULTS_DIR / "adapter_inventory.csv", index=False)
display(INVENTORY_DF.drop(columns=["adapter_model_sha256"]).round(4))


**Cell 26**

Summary of all complete datasets (recomputed from the adapters, also copied to Drive).


In [ ]:
# Cell 27
summary_rows = [dataset_geometry(key) for key in DATASET_ORDER if dataset_complete(key)]
SUMMARY_DF = pd.DataFrame(summary_rows).set_index("dataset")
SUMMARY_DF.to_csv(RESULTS_DIR / "summary_datasets.csv")
shutil.copy2(RESULTS_DIR / "summary_datasets.csv", GEOMETRY_DRIVE / "summary_datasets.csv")
display(SUMMARY_DF.round(4))


**Cell 28**

Comparison with NB14 (HelpSteer2, same five attributes, same recipe). Uses the NB14 geometry from the repository, or from the NB14 geometry ZIP on Drive; skipped if neither is found.


In [ ]:
# Cell 29
import io, zipfile
nb14_name = "nb14_llama2_7b_helpsteer2_dpo_seed137"
nb14_csv = PROJECT_ROOT / "results" / nb14_name / "geometry/R_llama2_dpo_cos.csv"
nb14_zip = Path("/content/drive/MyDrive/master-thesis-nb14") / nb14_name / f"{nb14_name}_geometry.zip"
R_hs2 = None
if nb14_csv.is_file():
    R_hs2 = pd.read_csv(nb14_csv, index_col=0)
elif nb14_zip.is_file():
    with zipfile.ZipFile(nb14_zip) as archive:
        member = f"results/{nb14_name}/geometry/R_llama2_dpo_cos.csv"
        R_hs2 = pd.read_csv(io.BytesIO(archive.read(member)), index_col=0)
if "helpsteer1" not in DATASET_ORDER or not dataset_complete("helpsteer1"):
    print("[skip] HelpSteer1 experts not complete")
elif R_hs2 is None:
    print("[skip] NB14 geometry not found")
else:
    attributes = OBJECTIVES["helpsteer1"]
    R_hs1 = pd.read_csv(RESULTS_DIR / "helpsteer1/R_cos.csv", index_col=0).loc[attributes, attributes]
    R_hs2 = R_hs2.loc[attributes, attributes]
    iu = np.triu_indices(len(attributes), 1)
    a, b = R_hs1.values[iu], R_hs2.values[iu]
    COMPARE_HS = {"pearson_offdiag": float(np.corrcoef(a, b)[0, 1]), "mean_abs_difference": float(np.abs(a - b).mean()),
                  "hs1_offdiag_mean": float(a.mean()), "hs2_offdiag_mean": float(b.mean())}
    (RESULTS_DIR / "comparison_helpsteer1_vs_helpsteer2.json").write_text(json.dumps(COMPARE_HS, indent=2) + "\n")
    print(json.dumps(COMPARE_HS, indent=2))
    display((R_hs1 - R_hs2).round(4))


**Cell 30**

Export. The geometry folder goes into the repository results; the finished adapters are bundled on Drive for later evaluation notebooks.


In [ ]:
# Cell 31
import zipfile
from datetime import datetime, timezone
from google.colab import files

RUN_MANIFEST = {
    "schema_version": 1, "notebook": "NB16 Llama-2-7B DPO experts on four preference datasets, seed 137",
    "created_utc": datetime.now(timezone.utc).isoformat(), "run_tag": RUN_TAG,
    "base_model": BASE_MODEL, "revisions": REV, "seed": SEED, "pair_seed": PAIR_SEED,
    "n_pairs_per_dataset": N_PAIRS, "max_pairs_cap": MAX_PAIRS_CAP, "same_n_all_datasets": SAME_N_ALL_DATASETS,
    "pair_selection": "one pair per prompt (NB12 select_pairs)",
    "eligible_prompts": {f"{k}/{o}": ELIGIBLE_PROMPTS[(k, o)] for k, o in EXPERTS},
    "humor_margin": HUMOR_MARGIN, "humor_scores_sha256": sha256_file(HUMOR_PATH),
    "prompt_format": "\\n\\nHuman: {prompt} " + RESPONSE_SPLIT.replace("\n", "\\n"),
    "training_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert_multidata.py"),
    "recipe_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert_llama2.py"),
    "pair_logic_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert_one_pair.py"),
    "nb11_helpers_script_sha256": sha256_file(PROJECT_ROOT / "scripts/0_dpo_expert.py"),
    "adapter_hashes": {f"{r['dataset']}/{r['objective']}": r["adapter_model_sha256"] for r in inventory},
    "pair_hashes": {f"{k}/{o}": PAIR_ID_HASHES[(k, o)] for k, o in EXPERTS},
    "candidate_counts": {f"{k}/{o}": CANDIDATE_COUNTS[(k, o)] for k, o in EXPERTS},
    "training_wall_clock_h": {k: v / 3600 for k, v in times.items()},
    "reward_model_used_during_training": False,
}
(RESULTS_DIR / "nb16_run_manifest.json").write_text(json.dumps(RUN_MANIFEST, indent=2) + "\n")

geometry_zip = Path(f"/content/{RUN_TAG}_geometry.zip")
with zipfile.ZipFile(geometry_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, f"results/{RUN_TAG}/geometry/{path.relative_to(RESULTS_DIR).as_posix()}")
shutil.copy2(geometry_zip, DRIVE_ROOT / geometry_zip.name)

adapter_zip = DRIVE_ROOT / f"{RUN_TAG}_adapters.zip"
with zipfile.ZipFile(adapter_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for key, objective in EXPERTS:
        if not is_complete(key, objective):
            continue
        root = expert_dir(key, objective)
        for path in sorted(root.rglob("*")):
            if path.is_file() and "trainer" not in path.parts:
                archive.write(path, f"{key}/dpo_{objective}/{path.relative_to(root).as_posix()}")
    archive.write(RESULTS_DIR / "nb16_run_manifest.json", "nb16_run_manifest.json")
print(f"geometry ZIP : {geometry_zip}\nadapter ZIP  : {adapter_zip} ({adapter_zip.stat().st_size / 1e6:.0f} MB)")
files.download(str(geometry_zip))
if DOWNLOAD_ADAPTER_BUNDLE:
    files.download(str(adapter_zip))


**Cell 32**

Repository status.


In [ ]:
# Cell 33
!git status --short
